# 🔬 HuggingFace TGI 架构深度剖析

> **核心命题**：作为 HuggingFace 的官方推理方案，TGI 如何在「兼容一切模型」和「极致性能」之间平衡？

TGI (Text Generation Inference) 是 HuggingFace 在 2022 年推出的推理服务方案。它的角色很微妙：
- 比 vLLM 更早提出 Continuous Batching 和 Prefix Caching
- 但后来在性能上被 vLLM 超越
- 然而它对 HuggingFace 模型的兼容性是最好的（不需要额外适配）

## 1. 架构全景

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│                        TGI 架构                               │
├──────────────────────────────────────────────────────────────────────────────────┤
│                                                               │
│  ┌─────────────────────────────────────────────────────┐ │
│  │                    Router (HTTP)                         │ │
│  │  • /generate            (TGI 原生 API)                   │ │
│  │  • /v1/chat/completions (OpenAI 兼容)                   │ │
│  │  • 请求校验 + Tokenization                              │ │
│  └─────────────────────────────────────────────────────┘ │
│                            │  gRPC (内部)                    │
│  ┌─────────────────────────────────────────────────────┐ │
│  │                   Scheduler                              │ │
│  │  • Continuous Batching (最早的生产级实现)                │ │
│  │  • Watermark-based 内存管理 (PagedAttention 之前)        │ │
│  │  • Speculative Decoding (Medusa / n-gram)               │ │
│  │  • Prefill / Decode 分离调度                             │ │
│  └─────────────────────────────────────────────────────┘ │
│                            │  shared memory / IPC            │
│  ┌─────────────────────────────────────────────────────┐ │
│  │                   Model Worker (Python)                  │ │
│  │  • HuggingFace Transformers 原生加载                     │ │
│  │  • FlashAttention v2 / PagedAttention 后端               │ │
│  │  • AWQ / GPTQ / EETQ / bitsandbytes 量化                │ │
│  │  • Tensor Parallelism (通过 accelerate)                  │ │
│  └─────────────────────────────────────────────────────┘ │
│                                                               │
└──────────────────────────────────────────────────────────────────────────────────┘
```

## 2. 独特的架构设计：Router + Worker 分离

TGI 最大的架构特点是将 HTTP 层和推理层分离为两个进程：

```rust
// Router (Rust 实现, ~2000 行)
// 负责: HTTP 服务 + 请求调度 + tokenization
// 核心结构:
struct Router {
    scheduler: Scheduler,      // 请求队列管理和 batch 组装
    tokenizer: Tokenizer,      // HuggingFace tokenizers (Rust 绑定)
    worker_client: WorkerRpc,  // gRPC 客户端，连接 Python worker
    state: ServerState,        // 全局状态
}

// Worker (Python 实现)
// 负责: 实际的模型推理
class ModelWorker:
    def __init__(self, model_id):
        self.model = AutoModelForCausalLM.from_pretrained(model_id)
        self.model = optimize(self.model)  # FlashAttention, quantization
        self.kv_cache = KVCacheManager()

    def forward(self, batch: Batch) -> Logits:
        return self.model(**batch.to_inputs())
```

**为什么用 Rust 写 Router？**
1. Tokenization 是 CPU intensive 的，Python 的 GIL 会阻塞 HTTP 处理
2. Rust 的 `tokenizers` 库（HuggingFace 出品）是原生的，比 Python 快 10×+
3. 调度逻辑简单但需要低延迟——Rust 没有 GC pauses
4. gRPC 通信让 Router 和 Worker 可以独立扩缩

## 3. Watermark Memory Management

在 vLLM 的 PagedAttention 之前，TGI 用的是「水位线」方案：

```python
# TGI 的水位线内存管理（已被 PagedAttention 取代）
class WatermarkKVCache:
    '''
    预分配一个大的 KV Cache buffer，用两个指针跟踪使用量。

    类比：一条固定长度的磁带。
    '''
    def __init__(self, total_capacity: int):
        self.buffer = allocate_kv_cache(total_capacity)
        self.current_watermark = 0  # 已经使用的 token 数
        self.freed_watermark = 0    # 已经完成并释放的 token 数

    def allocate_for_batch(self, num_tokens: int) -> int:
        '''为 batch 分配空间，返回起始位置'''
        if self.current_watermark + num_tokens > self.total_capacity:
            # 尝试从头部回收已完成请求的 KV Cache
            # 但如果还有请求占着头部，新的分配就会失败
            # ← 这是水位线方案的致命缺陷
            self.compact()

        start = self.current_watermark
        self.current_watermark += num_tokens
        return start

    def free(self, start: int, num_tokens: int):
        '''释放已完成请求的 KV Cache'''
        # 只能标记释放，不能真正移动数据
        # ← 产生碎片！
        pass

    def compact(self):
        '''压缩：把活跃的 KV blocks 移到 buffer 头部'''
        # 类似磁盘碎片整理 — 需要拷贝大量 KV Cache 数据
        # ← 耗时，但能恢复连续空间
        pass
```

TGI 后来也引入了 PagedAttention 后端，但默认配置仍然是 watermark 方案（稳定优先）。

## 4. Speculative Decoding：TGI 的杀手镖

TGI 对 Speculative Decoding 的支持是最成熟的。

```
普通自回归推理:
  Step 1: model(batch) → token_1
  Step 2: model(batch + [token_1]) → token_2
  Step 3: model(batch + [token_1, token_2]) → token_3
  问题：每步只生成 1 个 token，GPU 大部分时间在等显存读取

Speculative Decoding:
  用一个小模型（draft model）快速生成 K 个候选 token，
  然后用大模型一次性验证这 K 个 token。

  Step 1: draft_model(batch) → [t1′, t2′, t3′, t4′, t5′]  ← 快速生成 5 个
  Step 2: target_model(batch + [t1′, t2′, t3′, t4′, t5′]) → 一次 forward 验证
          结果: [t1′✓, t2′✓, t3′✗, t4, t5, t6]  ← 前 2 个正确，第 3 个修正后继续
  效果: 2 步生成 6 个 token，vs 普通需要 6 步 → 3× 加速

TGI 支持的方案:
  1. Medusa: 额外的「预测头」直接输出多个候选 token（无需 draft model）
  2. n-gram: 用最近的 n-gram 匹配做预测（零额外模型）
  3. 经典 draft-target: 用小模型做 draft
```

## 5. TGI vs vLLM vs TensorRT-LLM

| 维度 | TGI | vLLM | TensorRT-LLM |
|------|-----|------|-------------|
| **推出时间** | 2022 Q1 | 2023 Q2 | 2023 Q4 |
| **CB 成熟度** | 最早生产级实现 | 性能最好的实现 | In-flight 更先进 |
| **模型兼容性** | 最好（HuggingFace 原生） | 好（大多数架构） | 20+ 种模型 |
| **性能** | 中 | 高 | 最高 |
| **量化方案** | AWQ/GPTQ/bitsandbytes/EETQ | AWQ/GPTQ/FP8 | FP8/INT8/INT4 原生 |
| **Speculative Decoding** | 最好（Medusa + n-gram） | 实验性 | 不支持 |
| **硬件要求** | NVIDIA/AMD/Intel Gaudi | NVIDIA/AMD | NVIDIA only |
| **社区** | HuggingFace 官方 | 最活跃 | NVIDIA 官方 |
| **部署方式** | Docker / Inference Endpoints | pip install / Docker | 编译 + Docker |

**选 TGI 的场景**：
- 需要 HuggingFace 生态的模型（很多小众模型只有 TGI 能直接跑）
- 需要 Speculative Decoding（TGI 的实现最成熟）
- 用 HuggingFace Inference Endpoints（托管服务，零运维）
- 需要 Intel Gaudi / AMD GPU 支持

## 下一步

- → [04-tensorrt-llm-deep-dive.ipynb](04-tensorrt-llm-deep-dive.ipynb)：编译优化的极限
- → [../distributed/06-sglang-deep-dive.ipynb](../distributed/06-sglang-deep-dive.ipynb)：RadixAttention 的极致前缀复用
- → 返回 [../00-overview.ipynb](../00-overview.ipynb) 查看全景对比